In [14]:
'''
the Purpose of This Task
1. Accepts classical data
2. Encodes it into quantum states
3. Uses Qiskit’s Estimator to simulate quantum behavior
4. Connects to PyTorch for training
'''
# Load and Preprocess the Dataset
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load dataset
iris = load_iris()
X = iris.data[:, :2]  # Use only first two features
y = (iris.target != 0).astype(int)  # Binary classification: Setosa vs others

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

In [15]:
# Create a Parameterized Quantum Circuit
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# Define input parameters
x0 = Parameter("x0")
x1 = Parameter("x1")

# Create quantum circuit
qc = QuantumCircuit(2)
qc.h([0, 1])         # Put qubits into superposition
qc.rz(x0, 0)         # Encode feature x0
qc.rz(x1, 1)         # Encode feature x1

In [16]:
# Build the Quantum Neural Network (QNN)
from qiskit.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# Define QNN
estimator = Estimator()
qnn = EstimatorQNN(
    circuit=qc,
    input_params=[x0, x1],
    weight_params=[],  # No trainable weights in this simple version
    estimator=estimator
)

# Connect to PyTorch
model = TorchConnector(qnn)

C:\Users\All\AppData\Local\Temp\ipykernel_2128\977484787.py:7: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator()
C:\Users\All\AppData\Local\Temp\ipykernel_2128\977484787.py:8: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  qnn = EstimatorQNN(


In [17]:
# Train the Hybrid Model
import torch
import torch.nn as nn

# Prepare data
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

# Define loss and optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(100):
    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = loss_fn(output.squeeze(), y_train_tensor)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 0.7048
Epoch 10, Loss: 0.7048
Epoch 20, Loss: 0.7048
Epoch 30, Loss: 0.7048
Epoch 40, Loss: 0.7048
Epoch 50, Loss: 0.7048
Epoch 60, Loss: 0.7048
Epoch 70, Loss: 0.7048
Epoch 80, Loss: 0.7048
Epoch 90, Loss: 0.7048


In [18]:
# Evaluate Quantum vs Classical Performance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Classical model
clf = LogisticRegression()
clf.fit(X_train, y_train)
y_pred_classical = clf.predict(X_test)

# Quantum model
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_pred_quantum = model(X_test_tensor).detach().numpy().squeeze()
y_pred_quantum_binary = (y_pred_quantum > 0.5).astype(int)

# Print results
print("Classical Accuracy:", accuracy_score(y_test, y_pred_classical))
print("Quantum Accuracy:", accuracy_score(y_test, y_pred_quantum_binary))


Classical Accuracy: 1.0
Quantum Accuracy: 0.4222222222222222


In [ ]:
# What Do the Results Mean?
'''
Classical Accuracy: 1.0
Your logistic regression model nailed every prediction on the test set.
This means the classical model found a simple, linear decision boundary that perfectly separates Setosa from the other Iris species using just two features.

Quantum Accuracy: ~0.42
Your quantum model struggled—less than half of its predictions were correct.
Because the quantum circuit used was very shallow (just Hadamard + RZ gates) and had no trainable weights.

It encoded data but didn’t learn how to separate classes effectively.
'''
# To improve quantum accuracy
'''
- Add trainable weights (e.g., Ry(θ) gates)
- Introduce entanglement (e.g., CNOT gates)
- Use variational circuits and optimize them
'''